# Lab 5 · Red de incidencias, antenas, zonas y clientes afectados
### Tema 5 · MU Gestión de Sistemas Complejos · Caso transversal: operador de telecomunicaciones

**Entorno:** SageMaker Notebook (kernel Python 3) · **Librerías:** pandas, NetworkX, matplotlib
**Duración estimada:** 180 min · **Nivel:** intermedio

---

## 1. Idea general del laboratorio

Hasta ahora hemos mirado a la operadora **cliente a cliente**. Ese enfoque ignora algo esencial de un
sistema complejo: **su estructura**. Una operadora no es una lista de clientes independientes; es una
**red** física y lógica en la que clientes, antenas, zonas, nodos de agregación y núcleo están conectados,
y donde un fallo en un punto puede propagarse y afectar a miles de personas.

En este laboratorio representaremos esa estructura como un **grafo** (nodos unidos por aristas), la
construiremos con **NetworkX**, calcularemos **métricas estructurales** que revelan qué nodos son críticos,
mapearemos qué clientes dependen de cada nodo y cruzaremos la red con las **incidencias**. Es el cimiento
de los Labs 6 (resiliencia) y 7 (IA sobre grafos).

### Objetivos didácticos
- Explicar por qué un sistema complejo se representa como un grafo.
- Construir un grafo con NetworkX a partir de ficheros de nodos y aristas.
- Calcular e interpretar grado, intermediación, cercanía y puntos de articulación.
- Identificar nodos críticos y estimar cuántos clientes dependen de cada uno.
- Cruzar topología con incidencias para evaluar impacto sobre clientes.
- Visualizar la red y comunicar hallazgos a un público de gestión.

> **Nota para el vídeo.** Cada celda de código está comentada paso a paso para poder explicarla en
> pantalla: lee el comentario, ejecuta, y comenta el resultado obtenido.

## 2. Datos de partida

Cuatro CSV sintéticos con topología realista:

| Fichero | Filas | Contenido |
|---|---|---|
| `network_nodes.csv` | 50 | Nodos: id, tipo, región, zona, capacidad, coordenadas. |
| `network_edges.csv` | 66 | Enlaces: origen, destino, tipo, capacidad, distancia. |
| `customer_nodes.csv` | 800 | Cliente y nodo de acceso al que se conecta. |
| `network_incidents.csv` | 60 | Incidencias sobre nodos: tipo, severidad, duración. |

**Topología:** 2 nodos de núcleo redundantes, una agregación por región (6) y ≈42 accesos (antenas), con
anillos por región y algo de redundancia inter-región. Los clientes se modelan como **atributo** del nodo
de acceso, no como nodos del grafo.

In [ ]:
# ===========================================================================
# CELDA 1 · Instalación de dependencias
# ===========================================================================
# En una instancia nueva de SageMaker, NetworkX o matplotlib pueden no estar.
# Usamos {sys.executable} (la ruta del Python del kernel actual) en lugar de un
# "!pip install" pelado, para instalar en el entorno CORRECTO del notebook.
# La opcion -q (quiet) reduce el ruido de salida.
import sys
!{sys.executable} -m pip install -q networkx matplotlib pandas
print("Dependencias listas")

In [ ]:
# ===========================================================================
# CELDA 2 · Importar librerias y cargar los cuatro ficheros
# ===========================================================================
import pandas as pd             # tablas (DataFrames)
import networkx as nx           # construir y analizar grafos
import matplotlib.pyplot as plt # graficos / visualizacion de la red

# RUTA define DONDE estan los CSV. Por defecto '.' = la carpeta del notebook
# (lectura local, sin configurar nada). Para el flujo original con S3, cambia
# esta linea por:  RUTA = 's3://<tu-bucket>/graph'  (SageMaker lee S3 si el
# rol LabRole tiene permisos).
RUTA = '.'

# read_csv carga cada fichero en un DataFrame (tabla en memoria).
nodes = pd.read_csv(f'{RUTA}/network_nodes.csv')       # 50 nodos
edges = pd.read_csv(f'{RUTA}/network_edges.csv')       # 66 enlaces
cust  = pd.read_csv(f'{RUTA}/customer_nodes.csv')      # 800 clientes
inc   = pd.read_csv(f'{RUTA}/network_incidents.csv')   # 60 incidencias

# .shape = (filas, columnas): verifica que se cargaron las dimensiones esperadas.
print("nodes:", nodes.shape, "| edges:", edges.shape,
      "| customers:", cust.shape, "| incidents:", inc.shape)

# .head() muestra las primeras filas para inspeccionar las columnas reales.
nodes.head()

## Parte A · Construir el grafo con NetworkX

Grafo **no dirigido** (la fibra transmite en ambos sentidos): primero los nodos con sus atributos, luego
las aristas con los suyos.

In [ ]:
# ===========================================================================
# CELDA 3 · Crear el grafo: nodos + aristas
# ===========================================================================
# nx.Graph() crea un grafo NO dirigido y vacio (si quisieramos sentido en las
# conexiones usariamos nx.DiGraph; aqui la fibra es bidireccional).
G = nx.Graph()

# --- Anadir NODOS con atributos ---
# iterrows() recorre la tabla fila a fila y devuelve (indice, fila); usamos "_"
# para el indice porque no lo necesitamos.
for _, r in nodes.iterrows():
    # add_node(id, **atributos): el 1er argumento es el identificador unico; el
    # resto son atributos "pegados" al nodo, consultables luego con
    # G.nodes[id]['atributo'].
    G.add_node(r['node_id'],
               node_type=r['node_type'],    # 'core', 'aggregation' o 'access'
               region=r['region'],          # Madrid, Bilbao, ...
               zone=r['zone'],              # urbana, periurbana, rural
               capacity=r['capacity_gbps']) # capacidad del nodo (Gbps)

# --- Anadir ARISTAS con atributos ---
for _, r in edges.iterrows():
    # add_edge(a, b, **atributos): conecta 'source' con 'target'.
    G.add_edge(r['source'], r['target'],
               link_type=r['link_type'],    # backbone/core_uplink/access_ring/interconnect
               capacity=r['capacity_gbps'],
               distance=r['distance_km'])

# Comprobaciones del grafo recien construido:
print("Nodos:", G.number_of_nodes())     # esperado: 50
print("Enlaces:", G.number_of_edges())   # esperado: 66
# is_connected=True => todos los nodos forman UNA sola pieza (sin partes sueltas).
print("Conexo:", nx.is_connected(G))

### Paso A.3 · Asignar clientes a los nodos de acceso

Los clientes **no** son nodos del grafo, pero cada acceso "sostiene" a un número de clientes. Lo guardamos
como **atributo** del nodo: será clave para medir el impacto de un fallo.

In [ ]:
# ===========================================================================
# CELDA 4 · Contar clientes por nodo de acceso y guardarlo como atributo
# ===========================================================================
# groupby('access_node_id').size() agrupa los 800 clientes por su nodo de acceso
# y cuenta cuantos hay en cada grupo -> Serie {nodo: nº_clientes}.
clientes_por_nodo = cust.groupby('access_node_id').size()

# Recorremos TODOS los nodos y les anadimos el atributo 'n_clientes'.
for nid in G.nodes():
    # .get(nid, 0) devuelve el recuento, o 0 si el nodo no aparece en la tabla
    # de clientes (nucleos y agregaciones no tienen clientes directos -> 0).
    G.nodes[nid]['n_clientes'] = int(clientes_por_nodo.get(nid, 0))

# Los 5 accesos que mas clientes sostienen (los mas sensibles a un fallo).
# sorted(..., key=..., reverse=True) ordena de mayor a menor por 'n_clientes'.
top = sorted(G.nodes(data=True), key=lambda x: x[1]['n_clientes'], reverse=True)[:5]
for nid, attr in top:
    print(f"{nid:12s} {attr['node_type']:12s} {attr['n_clientes']:3d} clientes")

## Parte B · Métricas estructurales

### B.1 · Grado — ¿cuántas conexiones tiene?

In [ ]:
# ===========================================================================
# CELDA 5 · GRADO: numero de conexiones de cada nodo
# ===========================================================================
# G.degree() da pares (nodo, grado); dict(...) -> {nodo: grado}. Grado alto =
# muy conectado, pero OJO: estar muy conectado NO equivale a ser critico.
grado = dict(G.degree())

# pd.Series(...).sort_values(ascending=False): ordena de mayor a menor.
df_grado = pd.Series(grado).sort_values(ascending=False)
print(df_grado.head(8))

### B.2 · Intermediación (betweenness) — ¿es un punto de paso obligado?

In [ ]:
# ===========================================================================
# CELDA 6 · INTERMEDIACION: cuantas rutas cortas pasan por cada nodo
# ===========================================================================
# betweenness_centrality: para cada nodo, fraccion de caminos mas cortos (entre
# todos los pares) que lo atraviesan. Alto = nodo "puente" => cuello de botella.
bet = nx.betweenness_centrality(G)

df_bet = pd.Series(bet).sort_values(ascending=False)
print("Nodos mas criticos por intermediacion:")
for nid in df_bet.head(8).index:
    print(f'  {nid:12s} {G.nodes[nid]["node_type"]:12s} {df_bet[nid]:.3f}')

**Pregunta (vídeo).** ¿Encabezan núcleo y agregación? Eso confirma que son los puntos sensibles.

### B.3 · Cercanía (closeness) — ¿está cerca de todos?

In [ ]:
# ===========================================================================
# CELDA 7 · CERCANIA: como de cerca esta un nodo del resto de la red
# ===========================================================================
# closeness_centrality = inverso de la distancia media a todos los demas.
# Alta cercania => alcanza rapido al resto (tipico del nucleo).
clo = nx.closeness_centrality(G)
df_clo = pd.Series(clo).sort_values(ascending=False)
print(df_clo.head(6))

### B.4 · Puntos de articulación — ¿su caída parte la red?

In [ ]:
# ===========================================================================
# CELDA 8 · PUNTOS DE ARTICULACION: nodos cuya caida fragmenta la red
# ===========================================================================
# articulation_points devuelve los nodos "cuello": si se eliminan, el grafo se
# parte. list(...) materializa el generador en una lista.
articulacion = list(nx.articulation_points(G))
print("Puntos de articulacion:", len(articulacion))
for nid in articulacion:
    # Un punto de articulacion que ADEMAS sostiene muchos clientes es
    # doblemente peligroso: desconecta la red Y deja gente sin servicio.
    print(f'  {nid} ({G.nodes[nid]["node_type"]}) sostiene '
          f'{G.nodes[nid]["n_clientes"]} clientes directos')

## Parte C · Nodos críticos y clientes afectados

In [ ]:
# ===========================================================================
# CELDA 9 · Funcion: clientes que quedan AISLADOS si cae un nodo
# ===========================================================================
def clientes_afectados_si_cae(G, nodo, referencia="CORE-01"):
    # Trabajamos sobre una COPIA para no destruir el grafo original.
    H = G.copy()
    # Si cae justo la referencia de nucleo, usamos el otro nucleo como referencia.
    if nodo == referencia:
        referencia = "CORE-02"
    # Eliminamos el nodo: simulamos su caida.
    H.remove_node(nodo)

    afectados = 0
    # Para cada nodo con clientes, comprobamos si SIGUE teniendo camino al nucleo.
    for nid, attr in H.nodes(data=True):
        if attr['n_clientes'] > 0:
            # has_path=True si existe ruta; si NO hay ruta, esos clientes caen.
            if not nx.has_path(H, nid, referencia):
                afectados += attr['n_clientes']
    # Sumamos los clientes DIRECTOS del propio nodo caido.
    afectados += G.nodes[nodo]['n_clientes']
    return afectados

# Aplicamos a TODOS los nodos -> {nodo: clientes_afectados}.
impacto = {n: clientes_afectados_si_cae(G, n) for n in G.nodes()}
df_imp = pd.Series(impacto).sort_values(ascending=False)
print("Clientes afectados si cae cada nodo (top 10):")
print(df_imp.head(10))

In [ ]:
# ===========================================================================
# CELDA 10 · Tabla de criticidad combinada: estructura + impacto
# ===========================================================================
# Juntamos por nodo: tipo, intermediacion, clientes directos y clientes que
# aislaria su caida. Asi vemos que nodos son criticos por ESTRUCTURA y por
# IMPACTO a la vez.
resumen = pd.DataFrame({
    'tipo': {n: G.nodes[n]['node_type'] for n in G.nodes()},
    'betweenness': bet,
    'clientes_directos': {n: G.nodes[n]['n_clientes'] for n in G.nodes()},
    'clientes_afectados': impacto,
})
resumen = resumen.sort_values('clientes_afectados', ascending=False)
print(resumen.head(12))

## Parte D · Incidencias sobre la red

In [ ]:
# ===========================================================================
# CELDA 11 · Incidencias por tipo de nodo y severidad
# ===========================================================================
# merge() une incidencias con nodos por 'node_id', trayendo tipo y region.
# how='left' conserva todas las incidencias.
inc_nodo = inc.merge(nodes[['node_id', 'node_type', 'region']],
                     on='node_id', how='left')

# Conteo por combinacion (tipo de nodo, severidad).
tabla = inc_nodo.groupby(['node_type', 'severity']).size()
print(tabla)

In [ ]:
# ===========================================================================
# CELDA 12 · Impacto real: metrica "cliente x minuto"
# ===========================================================================
# A cada incidencia le asociamos los clientes que aislaria su nodo (.map cruza
# con el diccionario 'impacto' de la Parte C).
inc_nodo['clientes_afectados'] = inc_nodo['node_id'].map(impacto)

# "cliente x minuto" = clientes afectados * duracion. Metrica estandar de
# gravedad: una averia breve en un nodo critico puede pesar mas que una larga
# en uno periferico.
inc_nodo['impacto_cliente_minuto'] = (
    inc_nodo['clientes_afectados'] * inc_nodo['duration_minutes'])

top_inc = inc_nodo.sort_values("impacto_cliente_minuto", ascending=False)
print(top_inc[['incident_id', 'node_id', 'node_type', 'severity',
               'duration_minutes', 'clientes_afectados',
               'impacto_cliente_minuto']].head(10).to_string(index=False))

In [ ]:
# ===========================================================================
# CELDA 13 · Regiones mas castigadas por las incidencias
# ===========================================================================
# Agrupamos por region y agregamos: nº de incidencias, minutos de corte y el
# impacto cliente x minuto acumulado.
resumen_zona = inc_nodo.groupby('region').agg(
    incidencias=('incident_id', 'count'),
    minutos_totales=('duration_minutes', 'sum'),
    impacto=('impacto_cliente_minuto', 'sum')
).sort_values('impacto', ascending=False)
print(resumen_zona)

**Preguntas.** ¿Qué región acumula mayor impacto? ¿Coincide con las de peor calidad y churn (Labs 3-4)?

## Parte E · Visualización de la red

In [ ]:
# ===========================================================================
# CELDA 14 · Dibujo por tipo de nodo (tamano proporcional a clientes)
# ===========================================================================
colores = {'core': '#c0392b', 'aggregation': '#e67e22', 'access': '#2980b9'}
# Lista de colores en el MISMO orden que G.nodes() (debe coincidir el orden).
node_colors = [colores[G.nodes[n]['node_type']] for n in G.nodes()]
# Tamano = base 120 + 6 por cliente => los nodos con mas clientes se ven mayores.
node_sizes  = [120 + G.nodes[n]['n_clientes'] * 6 for n in G.nodes()]

plt.figure(figsize=(13, 9))
# spring_layout: modelo de fuerzas (los conectados se atraen). seed=42 fija la
# disposicion (reproducible); k controla la separacion entre nodos.
pos = nx.spring_layout(G, seed=42, k=0.5)
nx.draw_networkx_edges(G, pos, alpha=0.3)                                  # aristas semitransparentes
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes)
# Etiquetamos SOLO nucleo y agregacion para no saturar.
labels = {n: n for n in G.nodes()
          if G.nodes[n]['node_type'] in ('core', 'aggregation')}
nx.draw_networkx_labels(G, pos, labels, font_size=8)
plt.title('Red de la operadora: nucleo, agregacion y acceso')
plt.axis('off'); plt.tight_layout(); plt.show()

In [ ]:
# ===========================================================================
# CELDA 15 · Mapa de criticidad (color = intermediacion)
# ===========================================================================
# Ahora el COLOR codifica la intermediacion: cuanto mas rojo, mas critico.
# Reutilizamos la misma disposicion 'pos' de la figura anterior.
valores_bet = [bet[n] for n in G.nodes()]
plt.figure(figsize=(13, 9))
nx.draw_networkx_edges(G, pos, alpha=0.3)
# cmap='YlOrRd' = amarillo->naranja->rojo. Guardamos el objeto en 'nodos' para
# poder anadir la barra de color (colorbar).
nodos = nx.draw_networkx_nodes(G, pos, node_color=valores_bet,
                               cmap='YlOrRd', node_size=node_sizes)
plt.colorbar(nodos, label="Centralidad de intermediacion")
nx.draw_networkx_labels(G, pos, labels, font_size=8)
plt.title('Criticidad estructural de la red')
plt.axis('off'); plt.tight_layout(); plt.show()

> Para **guardar** la figura y entregarla: `plt.savefig("red.png", dpi=150, bbox_inches="tight")` antes de
> `plt.show()`.

## Parte F (opcional) · Amazon Neptune

NetworkX es ideal para grafos en memoria (decenas de nodos). Para redes de **millones de nodos**,
persistentes y concurrentes, conviene una BD de grafos gestionada como **Amazon Neptune**. En AWS Academy
puede no estar habilitado y genera coste por hora; aquí NetworkX basta.

## Parte G · Entregable · Parte H · Cierre

Entregable (5–7 págs.): red como grafo; métricas con interpretación; puntos de articulación y su riesgo;
tabla de criticidad combinada; análisis de incidencias (cliente × minuto y región más castigada); las dos
visualizaciones; y una **recomendación** (qué 3 nodos reforzarías primero), conectándola con el churn de
los Labs 3-4. Al terminar, **Stop** la instancia y **conserva los datos: el Lab 6 reutiliza esta red.**

---

### Cierre didáctico
El grafo es una herramienta de gestión que responde preguntas imposibles de contestar cliente a cliente.
El Lab 6 llevará esta red al límite simulando caídas; el Lab 7 aplicará IA.